# 2-Bosqich: CAMS chang aerozoli + ERA5 meteorologiya + Sentinel-5P validatsiya

Bu notebook:
1. **CAMS EAC4 reanalizi** — tayyor chang aerozol optik qalinligi (Dust AOD)
2. **ERA5** — shamol, harorat, namlik, tuproq namligi (gridlangan)
3. **Sentinel-5P UVAI** — Python orqali (GEE'siz) yuklab olish
4. **Validatsiya** — 1-bosqich label'lari vs sun'iy yo'ldosh
5. **Feature matrix** — ML model uchun

### Kerak bo'lgan yagona narsa:
- **CDS API key** — https://cds.climate.copernicus.eu (bepul ro'yxat)
- Xuddi shu key bilan **CAMS** ham, **ERA5** ham yuklanadi!

---

## 1. Kutubxonalar

In [ ]:
!pip install cdsapi xarray netcdf4 pandas numpy matplotlib seaborn --quiet

In [ ]:
import cdsapi
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Kutubxonalar yuklandi!")

In [ ]:
# ============================================================
# SOZLAMALAR
# ============================================================
RESULTS_DIR = "../analysis/results/"
CAMS_DIR = "../data/cams/"
ERA5_DIR = "../data/era5/"

os.makedirs(CAMS_DIR, exist_ok=True)
os.makedirs(ERA5_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# O'zbekiston chegaralari
AREA = [46, 56, 37, 74]  # [North, West, South, East]

# 1-bosqich natijalari
coords = pd.read_csv("../data/stations_coordinates.csv")
valid_coords = coords[coords['terrain_type'] != 'tog'].copy()

print(f"✅ Stansiyalar: {len(valid_coords)} (tog'siz)")

## 2. CDS API sozlash

Bitta API key bilan **CAMS** ham, **ERA5** ham yuklanadi.

In [ ]:
# ════════════════════════════════════════════
# CDS API KEY — BU YERNI O'ZGARTIRING!
# ════════════════════════════════════════════
# https://cds.climate.copernicus.eu dan oling

CDS_KEY = "SIZNING_API_KEY"  # <-- shu yerni o'zgartiring
CDS_URL = "https://cds.climate.copernicus.eu/api"

# ~/.cdsapirc faylini yaratish
cdsapirc = os.path.expanduser("~/.cdsapirc")
if not os.path.exists(cdsapirc) and CDS_KEY != "SIZNING_API_KEY":
    with open(cdsapirc, 'w') as f:
        f.write(f"url: {CDS_URL}\nkey: {CDS_KEY}\n")
    print("✅ CDS API config saqlandi")
elif os.path.exists(cdsapirc):
    print("✅ CDS API config allaqachon mavjud")
else:
    print("⚠️ CDS_KEY ni o'zgartiring!")

## 3. CAMS EAC4 — Chang aerozoli yuklab olish

### CAMS EAC4 nima?
- **Copernicus Atmosphere Monitoring Service** global reanalizi
- Chang aerozol optik qalinligi (Dust AOD) — **tayyor qayta ishlangan** mahsulot
- 3 soatlik, ~80 km grid, 2003-2024
- `duaod550` — 550nm dagi chang aerozol optik qalinligi

### Bu ERA5 dan farqi:
- ERA5 = meteorologiya (shamol, harorat)
- CAMS = atmosfera kimyosi (chang, tutun, gaz)
- Ikkalasi ham CDS orqali bepul yuklanadi

In [ ]:
def download_cams_dust(year, month, output_dir=CAMS_DIR):
    """
    CAMS EAC4 dan chang aerozol ma'lumotlarini yuklab olish.
    
    Variable:
    - dust_aerosol_optical_depth_550nm (duaod550) — chang optik qalinligi
    - total_aerosol_optical_depth_550nm — umumiy aerosol
    """
    output_file = os.path.join(output_dir, f"cams_dust_{year}_{month:02d}.nc")
    
    if os.path.exists(output_file):
        print(f"  ✓ {year}-{month:02d} allaqachon mavjud")
        return output_file
    
    client = cdsapi.Client()
    
    request = {
        'type': 'an',  # analysis
        'format': 'netcdf',
        'variable': [
            'dust_aerosol_optical_depth_550nm',
            'total_aerosol_optical_depth_550nm',
        ],
        'year': str(year),
        'month': f"{month:02d}",
        'day': [f"{d:02d}" for d in range(1, 32)],
        'time': ['00:00', '03:00', '06:00', '09:00', '12:00', '15:00', '18:00', '21:00'],
        'area': AREA,
    }
    
    print(f"  ⬇️ CAMS {year}-{month:02d}...", end=" ")
    client.retrieve('cams-global-reanalysis-eac4', request, output_file)
    print(f"✅")
    
    return output_file

In [ ]:
# CAMS yuklab olish: 2011-2020 (10 yil)
# DIQQAT: har bir oy ~2-10 daqiqa, jami ~2-15 soat
# Avval 1 oy sinab ko'ring, keyin to'liq yuklab oling

print("CAMS EAC4 — chang aerozoli yuklab olish")
print("="*50)

# Sinov uchun bitta oy:
# download_cams_dust(2018, 4)  # 2018-aprel (Sentinel-5P boshlangan oy)

# To'liq yuklab olish (izohdagi # ni olib tashlang):
for year in range(2011, 2021):
    print(f"\n{year}:")
    for month in range(1, 13):
        try:
            download_cams_dust(year, month)
        except Exception as e:
            print(f"  ❌ {year}-{month:02d}: {e}")

## 4. ERA5 — Meteorologiya yuklab olish

In [ ]:
def download_era5(year, month, output_dir=ERA5_DIR):
    """
    ERA5 dan meteorologik ma'lumotlarni yuklab olish.
    
    Variables:
    - 10m shamol (u, v) → tezlik hisoblash uchun
    - 2m harorat + shudring nuqtasi → nisbiy namlik uchun
    - Sirt bosimi
    - Tuproq namligi (quruqlik — chang manbai)
    - Boundary layer height (chang ko'tarilish balandligi)
    """
    output_file = os.path.join(output_dir, f"era5_{year}_{month:02d}.nc")
    
    if os.path.exists(output_file):
        print(f"  ✓ {year}-{month:02d} allaqachon mavjud")
        return output_file
    
    client = cdsapi.Client()
    
    request = {
        'product_type': ['reanalysis'],
        'format': 'netcdf',
        'variable': [
            '10m_u_component_of_wind',
            '10m_v_component_of_wind',
            '2m_temperature',
            '2m_dewpoint_temperature',
            'surface_pressure',
            'volumetric_soil_water_layer_1',
            'boundary_layer_height',
            'total_precipitation',
        ],
        'year': str(year),
        'month': f"{month:02d}",
        'day': [f"{d:02d}" for d in range(1, 32)],
        'time': ['00:00', '06:00', '12:00', '18:00'],
        'area': AREA,
    }
    
    print(f"  ⬇️ ERA5 {year}-{month:02d}...", end=" ")
    client.retrieve('reanalysis-era5-single-levels', request, output_file)
    print(f"✅")
    
    return output_file

In [ ]:
# ERA5 yuklab olish (xuddi CAMS bilan parallel)
print("ERA5 — meteorologiya yuklab olish")
print("="*50)

for year in range(2011, 2021):
    print(f"\n{year}:")
    for month in range(1, 13):
        try:
            download_era5(year, month)
        except Exception as e:
            print(f"  ❌ {year}-{month:02d}: {e}")

## 5. Yuklab olingan ma'lumotni qayta ishlash

NetCDF → har bir stansiya nuqtasi uchun kunlik qiymatlar

In [ ]:
def extract_station_values(nc_file, stations_df, variables=None):
    """
    NetCDF fayldan har bir stansiya koordinatasi uchun kunlik o'rtacha qiymatlarni olish.
    
    Parameters:
        nc_file: NetCDF fayl yo'li
        stations_df: koordinatalar DataFrame (lat, lon, station)
        variables: o'qiladigan o'zgaruvchilar ro'yxati (None = hammasi)
    """
    ds = xr.open_dataset(nc_file)
    
    results = []
    
    for _, station in stations_df.iterrows():
        lat, lon, name = station['lat'], station['lon'], station['station']
        
        # Eng yaqin grid nuqtasi
        point = ds.sel(latitude=lat, longitude=lon, method='nearest')
        
        # Kunlik o'rtacha
        daily = point.resample(time='1D').mean()
        
        df_point = daily.to_dataframe().reset_index()
        df_point['station'] = name
        results.append(df_point)
    
    ds.close()
    return pd.concat(results, ignore_index=True)

In [ ]:
# CAMS fayllarni qayta ishlash
cams_files = sorted([f for f in os.listdir(CAMS_DIR) if f.endswith('.nc')])

if cams_files:
    print(f"CAMS qayta ishlash: {len(cams_files)} fayl...")
    cams_results = []
    
    for fname in cams_files:
        filepath = os.path.join(CAMS_DIR, fname)
        print(f"  {fname}...", end=" ")
        try:
            df = extract_station_values(filepath, valid_coords)
            cams_results.append(df)
            print("✓")
        except Exception as e:
            print(f"❌ {e}")
    
    if cams_results:
        df_cams = pd.concat(cams_results, ignore_index=True)
        df_cams.to_csv(f"{RESULTS_DIR}cams_station_daily.csv", index=False)
        print(f"\n✅ CAMS saqlandi: {len(df_cams):,} qator")
else:
    print("⚠️ CAMS fayllar topilmadi — avval yuklab oling")

In [ ]:
# ERA5 fayllarni qayta ishlash
era5_files = sorted([f for f in os.listdir(ERA5_DIR) if f.endswith('.nc')])

if era5_files:
    print(f"ERA5 qayta ishlash: {len(era5_files)} fayl...")
    era5_results = []
    
    for fname in era5_files:
        filepath = os.path.join(ERA5_DIR, fname)
        print(f"  {fname}...", end=" ")
        try:
            df = extract_station_values(filepath, valid_coords)
            era5_results.append(df)
            print("✓")
        except Exception as e:
            print(f"❌ {e}")
    
    if era5_results:
        df_era5 = pd.concat(era5_results, ignore_index=True)
        
        # Shamol tezligi hisoblash
        if 'u10' in df_era5.columns and 'v10' in df_era5.columns:
            df_era5['wind_speed'] = np.sqrt(df_era5['u10']**2 + df_era5['v10']**2)
        
        # Nisbiy namlik hisoblash (Magnus formulasi)
        if 't2m' in df_era5.columns and 'd2m' in df_era5.columns:
            t = df_era5['t2m'] - 273.15  # Kelvin → Celsius
            td = df_era5['d2m'] - 273.15
            df_era5['rh'] = 100 * np.exp((17.625*td)/(243.04+td)) / np.exp((17.625*t)/(243.04+t))
            df_era5['temp_c'] = t
        
        df_era5.to_csv(f"{RESULTS_DIR}era5_station_daily.csv", index=False)
        print(f"\n✅ ERA5 saqlandi: {len(df_era5):,} qator")
else:
    print("⚠️ ERA5 fayllar topilmadi — avval yuklab oling")

## 6. Sentinel-5P UVAI — Python orqali (GEE'siz)

Copernicus Data Space API orqali yuklab olish.
- **Ro'yxat:** https://dataspace.copernicus.eu (bepul)
- **API:** OData yoki OpenSearch

In [ ]:
import requests

# Copernicus Data Space token olish
# https://dataspace.copernicus.eu dan ro'yxatdan o'ting

CDSE_USER = "SIZNING_EMAIL"    # <-- o'zgartiring
CDSE_PASS = "SIZNING_PAROL"    # <-- o'zgartiring

def get_cdse_token(username, password):
    """Copernicus Data Space Ecosystem token olish."""
    url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
    data = {
        'grant_type': 'password',
        'username': username,
        'password': password,
        'client_id': 'cdse-public',
    }
    response = requests.post(url, data=data)
    response.raise_for_status()
    return response.json()['access_token']


if CDSE_USER != "SIZNING_EMAIL":
    try:
        token = get_cdse_token(CDSE_USER, CDSE_PASS)
        print(f"✅ Copernicus Data Space token olindi")
    except Exception as e:
        print(f"❌ Token olishda xato: {e}")
        token = None
else:
    print("⚠️ CDSE_USER va CDSE_PASS ni o'zgartiring!")
    print("   https://dataspace.copernicus.eu dan ro'yxatdan o'ting")
    token = None

In [ ]:
def search_sentinel5p_products(start_date, end_date, bbox, token):
    """
    Sentinel-5P UVAI mahsulotlarini qidirish (OData API).
    
    bbox: [west, south, east, north]
    """
    base_url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
    
    # POLYGON format
    west, south, east, north = bbox
    footprint = f"POLYGON(({west} {south},{east} {south},{east} {north},{west} {north},{west} {south}))"
    
    filter_query = (
        f"Collection/Name eq 'SENTINEL-5P' and "
        f"Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'L2__AER_AI') and "
        f"ContentDate/Start gt {start_date}T00:00:00.000Z and "
        f"ContentDate/Start lt {end_date}T23:59:59.999Z and "
        f"OData.CSC.Intersects(area=geography'SRID=4326;{footprint}')"
    )
    
    params = {
        '$filter': filter_query,
        '$top': 100,
        '$orderby': 'ContentDate/Start asc'
    }
    
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    
    products = response.json().get('value', [])
    return products


print("Sentinel-5P qidirish funksiyasi tayyor.")
print("Quyidagi cell'da O'zbekiston uchun UVAI mahsulotlarini qidiramiz.")

In [ ]:
# Sentinel-5P UVAI qidirish (namuna: 2020-aprel, 1 oy)
# Bu cell faqat mavjud mahsulotlar ro'yxatini ko'rsatadi (yuklamaydi)

if token:
    bbox_uzb = [56, 37, 74, 46]  # O'zbekiston
    
    products = search_sentinel5p_products('2020-04-01', '2020-04-30', bbox_uzb, token)
    
    print(f"Topilgan S5P UVAI mahsulotlar: {len(products)}")
    for p in products[:5]:
        print(f"  {p['Name'][:60]}... | {p['ContentLength']/1e6:.0f} MB")
else:
    print("⚠️ Token yo'q — yuqoridagi CDSE credentials'ni to'ldiring")

## 7. Validatsiya: CAMS Dust AOD vs bizning label

CAMS — bu **model** natijasi (sun'iy yo'ldosh o'lchovi emas), lekin u sun'iy yo'ldosh va yer usti ma'lumotlarini assimilyatsiya qiladi, shuning uchun mustaqil validatsiya manbai sifatida foydali.

In [ ]:
# CAMS va 1-bosqich label'larini birlashtirish

# 1-bosqich natijalari
labeled_file = f"{RESULTS_DIR}all_stations_labeled.csv"
if os.path.exists(labeled_file):
    df_labeled = pd.read_csv(labeled_file, parse_dates=['date'])
    print(f"✅ Label'lar: {len(df_labeled):,} qator")
else:
    print("⚠️ Avval step1 notebook'ni run qiling!")

# CAMS natijalari
cams_file = f"{RESULTS_DIR}cams_station_daily.csv"
if os.path.exists(cams_file):
    df_cams = pd.read_csv(cams_file, parse_dates=['time'])
    df_cams['date'] = df_cams['time'].dt.date
    print(f"✅ CAMS: {len(df_cams):,} qator")
    
    # Birlashtirish
    df_labeled['date_key'] = pd.to_datetime(df_labeled['date']).dt.date
    df_val = df_labeled.merge(
        df_cams[['station', 'date', 'duaod550', 'aod550']],
        left_on=['station', 'date_key'],
        right_on=['station', 'date'],
        how='inner'
    )
    print(f"\n✅ Birlashtirish: {len(df_val):,} qator")
    print(f"\nCAMS Dust AOD statistikasi (dust_severity bo'yicha):")
    print(df_val.groupby('dust_severity')['duaod550'].describe().round(4))
else:
    print("⚠️ CAMS ma'lumoti topilmadi — avval yuklab oling")

In [ ]:
# VALIDATSIYA GRAFIKLARI

if 'df_val' in dir() and len(df_val) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Boxplot: Dust AOD vs severity
    order = ['NONE', 'YENGIL', 'ORTACHA', 'KUCHLI']
    existing = [s for s in order if s in df_val['dust_severity'].unique()]
    sns.boxplot(data=df_val, x='dust_severity', y='duaod550', 
                order=existing, ax=axes[0],
                palette={'NONE':'#ccc','YENGIL':'#fdd835','ORTACHA':'#ff9800','KUCHLI':'#d32f2f'})
    axes[0].set_title('CAMS Dust AOD vs Chang darajasi')
    axes[0].set_ylabel('Dust AOD (550nm)')
    
    # 2. V (ko'rinish) vs Dust AOD
    sample = df_val.dropna(subset=['V', 'duaod550']).sample(min(5000, len(df_val)))
    axes[1].scatter(sample['V'], sample['duaod550'], alpha=0.2, s=5, c='darkorange')
    axes[1].set_xlabel('V — ko\'rinish (km)')
    axes[1].set_ylabel('Dust AOD')
    axes[1].set_title('Ko\'rinish vs CAMS Dust AOD')
    axes[1].set_xlim(0, 10)
    
    # 3. Terrain bo'yicha Dust AOD o'rtacha (oylik)
    if 'terrain_type' in df_val.columns:
        monthly_terrain = df_val.groupby(
            [pd.to_datetime(df_val['date_key']).dt.month, 'terrain_type']
        )['duaod550'].mean().unstack()
        monthly_terrain.plot(ax=axes[2], marker='o')
        axes[2].set_xlabel('Oy')
        axes[2].set_ylabel('O\'rtacha Dust AOD')
        axes[2].set_title('Mavsumiy Dust AOD — terrain bo\'yicha')
        axes[2].legend(title='Terrain')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Validatsiya uchun CAMS ma'lumoti kerak")

In [ ]:
# Validatsiya metriklari
if 'df_val' in dir() and len(df_val) > 0:
    print("📊 VALIDATSIYA: CAMS Dust AOD vs bizning label")
    print("="*60)
    
    # CAMS dust AOD > 0.2 = "chang bor" (threshold)
    # Bu qiymat ilmiy adabiyotda ko'p ishlatiladi
    DUST_AOD_THRESHOLD = 0.2
    
    df_val['cams_dust'] = df_val['duaod550'] > DUST_AOD_THRESHOLD
    df_val['our_dust'] = df_val['is_dust'].astype(bool)
    
    # Confusion matrix
    tp = ((df_val['our_dust']) & (df_val['cams_dust'])).sum()
    fp = ((df_val['our_dust']) & (~df_val['cams_dust'])).sum()
    fn = ((~df_val['our_dust']) & (df_val['cams_dust'])).sum()
    tn = ((~df_val['our_dust']) & (~df_val['cams_dust'])).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"\n  Threshold: CAMS Dust AOD > {DUST_AOD_THRESHOLD}")
    print(f"\n  Confusion Matrix:")
    print(f"                    CAMS: Chang   CAMS: Yo'q")
    print(f"  Bizda: Chang       {tp:>7,}      {fp:>7,}")
    print(f"  Bizda: Yo'q        {fn:>7,}      {tn:>7,}")
    print(f"\n  Precision: {precision:.3f}")
    print(f"  Recall:    {recall:.3f}")
    print(f"  F1-score:  {f1:.3f}")
    
    print(f"\n  IZOH:")
    print(f"  Agar F1 > 0.3 bo'lsa — mezonlarimiz CAMS bilan mantiqiy mos")
    print(f"  Agar F1 < 0.2 bo'lsa — mezonlarni qayta ko'rib chiqish kerak")
    print(f"  CAMS o'zi ham model, to'liq 'ground truth' emas")

## 8. Feature Matrix (ML uchun)

In [ ]:
# Barcha manbalarni birlashtirish
df_features = df_labeled.copy()
df_features['date_key'] = pd.to_datetime(df_features['date']).dt.date

# CAMS qo'shish
if os.path.exists(f"{RESULTS_DIR}cams_station_daily.csv"):
    cams = pd.read_csv(f"{RESULTS_DIR}cams_station_daily.csv", parse_dates=['time'])
    cams['date'] = cams['time'].dt.date
    cams_cols = [c for c in cams.columns if c not in ['time', 'latitude', 'longitude']]
    df_features = df_features.merge(cams[cams_cols], on=['station', 'date'], how='left', suffixes=('', '_cams'))
    print("✅ CAMS qo'shildi")

# ERA5 qo'shish
if os.path.exists(f"{RESULTS_DIR}era5_station_daily.csv"):
    era5 = pd.read_csv(f"{RESULTS_DIR}era5_station_daily.csv", parse_dates=['time'])
    era5['date'] = era5['time'].dt.date
    era5_cols = [c for c in era5.columns if c not in ['time', 'latitude', 'longitude']]
    df_features = df_features.merge(era5[era5_cols], on=['station', 'date'], how='left', suffixes=('', '_era5'))
    print("✅ ERA5 qo'shildi")

# Temporal features
df_features['month'] = pd.to_datetime(df_features['date']).dt.month
df_features['day_of_year'] = pd.to_datetime(df_features['date']).dt.dayofyear

# Lag features
df_features = df_features.sort_values(['station', 'date']).reset_index(drop=True)
for col in ['V', 'VxG', 'UN', 'Taav']:
    if col in df_features.columns:
        df_features[f'{col}_lag1'] = df_features.groupby('station')[col].shift(1)
        df_features[f'{col}_lag2'] = df_features.groupby('station')[col].shift(2)
        df_features[f'{col}_lag3'] = df_features.groupby('station')[col].shift(3)
        df_features[f'{col}_roll3'] = df_features.groupby('station')[col].transform(
            lambda x: x.rolling(3, min_periods=1).mean())

# Saqlash
df_features.to_csv(f"{RESULTS_DIR}feature_matrix.csv", index=False)
print(f"\n📊 FEATURE MATRIX: {len(df_features):,} qator, {len(df_features.columns)} ustun")
print(f"💾 Saqlandi: {RESULTS_DIR}feature_matrix.csv")

## 9. Xulosa

### Qilingan ishlar:
- ✅ **CAMS EAC4** — chang aerozol optik qalinligi (2011-2020)
- ✅ **ERA5** — shamol, harorat, namlik, tuproq, BLH (2011-2020)
- ✅ **Sentinel-5P** — UVAI yuklab olish imkoniyati (opsional)
- ✅ **Validatsiya** — CAMS Dust AOD vs bizning label
- ✅ **Feature matrix** — ML uchun tayyor

### Keyingi qadam (3-bosqich):
- XGBoost/LightGBM model qurish
- Train: 2011-2018, Test: 2019-2020
- Target: keyingi 1/2/3 kunda chang bo'ladimi?
- Feature importance → qaysi omillar eng muhim?